- notebook: [example][1]
- dataset: [Flowers Recognition][2]

[1]: https://www.kaggle.com/code/aritrag/simclr
[2]: https://www.kaggle.com/datasets/alxmamaev/flowers-recognition


# 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from torchvision.datasets import ImageFolder
from torchvision.transforms import (
    RandomResizedCrop,
    RandomHorizontalFlip,
    ColorJitter,
    RandomGrayscale,
    RandomApply,
    Compose,
    GaussianBlur,
    ToTensor,
)
import torchvision.models as models

import os
import glob
import time
from skimage import io
import matplotlib.pyplot as plt

In [ ]:
print(f'Torch-Version {torch.__version__}')
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f'DEVICE: {DEVICE}')

# 2. Stochastic Augmentation Module

In [ ]:
# Lives in dataset.py so DataLoader spawn workers can import it.
from dataset import (
    get_complete_transform,
    ContrastiveLearningViewGenerator,
)


# 3. Prepare the Data

In [ ]:
# Lives in dataset.py so DataLoader spawn workers can import it.
from dataset import CustomDataset


In [ ]:
# The size of the images
output_shape = [224,224]
kernel_size = [21,21] # 10% of the output_shape

# The custom transform
base_transforms = get_complete_transform(output_shape=output_shape, kernel_size=kernel_size, s=1.0)
custom_transform = ContrastiveLearningViewGenerator(base_transform=base_transforms)

flowers_ds = CustomDataset(
    list_images=glob.glob("./data/flowers/*/*.jpg"),
    transform=custom_transform
)

In [ ]:
# Visualise the data pipeline
plt.figure(figsize=(10,20))
def view_data(flowers, index):
    for i in range(1,6):
        images = flowers[index]
        view1, view2 = images
        plt.subplot(5,2,2*i-1)
        plt.imshow(view1.permute(1,2,0))
        plt.subplot(5,2,2*i)
        plt.imshow(view2.permute(1,2,0))

len(flowers_ds[2000])
# view_data(flowers_ds,2000)
print(f"Number of images in flowers_ds: {len(flowers_ds)}")




In [ ]:
BATCH_SIZE = 128

# dataset.py is importable by spawn workers, so multiprocessing can stay on.
# NUM_WORKERS = os.cpu_count()
NUM_WORKERS = 2

# Building the data loader
train_dl = torch.utils.data.DataLoader(
    flowers_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    drop_last=True,
    pin_memory=False,
)


# 4. SimCLR Model

In [ ]:
class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()
    def forward(self, x):
        return x

class SimCLR(nn.Module):
    def __init__(self, linear_eval=False):
        super().__init__()
        self.linear_eval = linear_eval
        resnet18 = models.resnet18(pretrained=False)
        resnet18.fc = Identity()
        self.encoder = resnet18
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 256)
        )
    def forward(self, x):
        if not self.linear_eval:
            x = torch.cat(x, dim=0)
        
        encoding = self.encoder(x)
        projection = self.projection(encoding) 
        return projection


# 5. Contrastive Loss

In [ ]:
LABELS = torch.cat([torch.arange(BATCH_SIZE) for i in range(2)], dim=0)
LABELS = (LABELS.unsqueeze(0) == LABELS.unsqueeze(1)).float() # Creates a one-hot with broadcasting
LABELS = LABELS.to(DEVICE) #128,128

def cont_loss(features, temp):
    """
    The NTxent Loss.
    
    Args:
        z1: The projection of the first branch
        z2: The projeciton of the second branch
    
    Returns:
        the NTxent loss
    """
    similarity_matrix = torch.matmul(features, features.T) # 128, 128
    # discard the main diagonal from both: labels and similarities matrix
    mask = torch.eye(LABELS.shape[0], dtype=torch.bool).to(DEVICE)
    # ~mask is the negative of the mask
    # the view is required to bring the matrix back to shape
    labels = LABELS[~mask].view(LABELS.shape[0], -1) # 128, 127
    similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1) # 128, 127

    # select and combine multiple positives
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1) # 128, 1

    # select only the negatives
    negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1) # 128, 126

    logits = torch.cat([positives, negatives], dim=1) # 128, 127
    labels = torch.zeros(logits.shape[0], dtype=torch.long).to(DEVICE)

    logits = logits / temp
    return logits, labels


In [ ]:
simclr_model = SimCLR().to(DEVICE)
criterion = nn.CrossEntropyLoss().to(DEVICE)
optimizer = torch.optim.Adam(simclr_model.parameters())


In [ ]:
%%time

EPOCHS = 10
for epoch in range(EPOCHS):
    t0 = time.time()
    running_loss = 0.0
    for i, views in enumerate(train_dl):
        projections = simclr_model([view.to(DEVICE) for view in views])
        logits, labels = cont_loss(projections, temp=2)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # print statistics
        running_loss += loss.item()
        if i % 10 == 9:    # print every 10 mini-batches
            print(f'EPOCH: {epoch+1} BATCH: {i+1} LOSS: {(running_loss/100):.4f} ')
            running_loss = 0.0
    print(f'Time taken: {((time.time()-t0)/60):.3f} mins')


# 6. Down Stream Task

In [ ]:
from torchvision.transforms import (
    CenterCrop,
    Resize
)


In [23]:
resize = Resize(255)
ccrop = CenterCrop(224)
ttensor = ToTensor()

custom_transform = Compose([
    resize,
    ccrop,
    ttensor,
])

flowers_ds = ImageFolder(
    root="./data/flowers/",
    transform=custom_transform
)

nu_classes = len(flowers_ds.classes)

BATCH_SIZE = 128

NUM_WORKERS=2
# NUM_WORKERS=os.cpu_count()

# Building the data loader
train_dl = torch.utils.data.DataLoader(
    flowers_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    drop_last=True,
    pin_memory=True,
)

In [24]:
class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()
    def forward(self, x):
        return x

class LinearEvaluation(nn.Module):
    def __init__(self, model, nu_classes):
        super().__init__()
        simclr = model
        simclr.linear_eval = True
        simclr.projection = Identity()
        self.simclr = simclr
        for param in self.simclr.parameters():
            param.requires_grad = False
        self.linear = nn.Linear(512, nu_classes)
    def forward(self, x):
        encoding = self.simclr(x)
        pred = self.linear(encoding) 
        return pred


In [25]:
eval_model = LinearEvaluation(simclr_model, nu_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss().to(DEVICE)
optimizer = torch.optim.Adam(eval_model.parameters())


In [26]:
EPOCHS = 10
for epoch in range(EPOCHS):
    t0 = time.time()
    running_loss = 0.0
    for i, element in enumerate(train_dl):
        image, label = element
        image = image.to(DEVICE)
        label = label.to(DEVICE)
        pred = eval_model(image)
        loss = criterion(pred, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # print statistics
        running_loss += loss.item()
        if i % 10 == 9:    # print every 10 mini-batches
            print(f'EPOCH: {epoch+1} BATCH: {i+1} LOSS: {(running_loss/100):.4f} ')
            running_loss = 0.0
    print(f'Time taken: {((time.time()-t0)/60):.3f} mins')


/Users/karlxu/workspace/github/xuyuji9000/machine-learning-playground/01_tasks/01_simclr_3rd-attempt/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


EPOCH: 1 BATCH: 10 LOSS: 0.1553 
EPOCH: 1 BATCH: 20 LOSS: 0.1403 
EPOCH: 1 BATCH: 30 LOSS: 0.1318 
Time taken: 0.249 mins
EPOCH: 2 BATCH: 10 LOSS: 0.1287 
EPOCH: 2 BATCH: 20 LOSS: 0.1232 
EPOCH: 2 BATCH: 30 LOSS: 0.1208 
Time taken: 0.245 mins
EPOCH: 3 BATCH: 10 LOSS: 0.1208 
EPOCH: 3 BATCH: 20 LOSS: 0.1203 
EPOCH: 3 BATCH: 30 LOSS: 0.1166 
Time taken: 0.247 mins
EPOCH: 4 BATCH: 10 LOSS: 0.1176 
EPOCH: 4 BATCH: 20 LOSS: 0.1169 
EPOCH: 4 BATCH: 30 LOSS: 0.1195 
Time taken: 0.245 mins
EPOCH: 5 BATCH: 10 LOSS: 0.1152 
EPOCH: 5 BATCH: 20 LOSS: 0.1139 
EPOCH: 5 BATCH: 30 LOSS: 0.1164 
Time taken: 0.248 mins
EPOCH: 6 BATCH: 10 LOSS: 0.1159 
EPOCH: 6 BATCH: 20 LOSS: 0.1161 
EPOCH: 6 BATCH: 30 LOSS: 0.1133 
Time taken: 0.246 mins
EPOCH: 7 BATCH: 10 LOSS: 0.1140 
EPOCH: 7 BATCH: 20 LOSS: 0.1119 
EPOCH: 7 BATCH: 30 LOSS: 0.1108 
Time taken: 0.247 mins
EPOCH: 8 BATCH: 10 LOSS: 0.1124 
EPOCH: 8 BATCH: 20 LOSS: 0.1111 
EPOCH: 8 BATCH: 30 LOSS: 0.1153 
Time taken: 0.245 mins
EPOCH: 9 BATCH: 10 LOSS: